# exp-018: 학습 가능한 fusion head (V1 — 관점별 선형 head)

- **목적:** 5관점 fusion 가중치를 *임의값*에서 **데이터로 학습한 값**으로 대체. V1(관점별 선형 Ridge)이 임의 가중치를 능가하고 황금 가중치(grid)에 필적하는지 검증. (RQ3 / 차별성 ④ GT부재·⑤ End-to-End LLM 재정의)
- **차별성 축:** ② 한국어, ④ GT 부재(LLM 생성 graded relevance), ⑤ 매칭 head 자체 학습(LLM 합성·재랭킹 0건)
- **입력 데이터:** `raw/data/user_data.csv` (999 user / 11,986 자소서) × `raw/data/company_jobdescription_enriched.partial.csv` (3,000 JD)
- **출력 위치:** `raw/experiments/exp-018-learnable-fusion-head/`
- **관련 위키:** learnable-fusion-실험계획 §4 exp-018, exp-017-validation-grid-search
- **작성일:** 2026-05-31
- **시드:** 42
- **LLM 호출:** ❌ 0건 (임베딩 백본은 V2/V3에서만, V1은 규칙 컴포넌트만 사용)

## 설계 결정 (2026-05-31 사용자 확정)

1. **후보 풀 = 주신 3,000 JD 전체.** 평가는 각 held-out user를 3,000 JD에 랭킹 → exp-017의 NDCG@10=1.0 천장 해소.
2. **라벨 = LLM가 데이터를 보고 직접 생성한 graded relevance(0~4).** 규칙·LLM-API 아님. 관점별 가중 + 비선형 상호작용(off-target 페널티, double-match 보너스).
3. **순환성 완화:** (a) head 입력 5컴포넌트보다 라벨러가 더 풍부(역량매칭 + 비선형) → 선형 head가 못 따라잡는 여지 = V2/V3 headroom, (b) **userId 기준 train 800 / held-out 199 분리**, (c) arbitrary·golden·learned 3자 비교.
4. **컴포넌트(5, distinct):** role_match / hard_skill / industry_match / star_overlap / competency. (원 exp-017의 `context`는 산업과 중복이라 제거하고 `competency`=자소서 역량키워드↔jd_competencies 추가.)

In [1]:
import pandas as pd, numpy as np, json, re, ast, time
from pathlib import Path
from sklearn.linear_model import Ridge

ROOT = Path('.')
RAW = ROOT / 'raw' / 'data'
OUT_DIR = ROOT / 'raw' / 'experiments' / 'exp-018-learnable-fusion-head'
OUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(42); rng = np.random.default_rng(42)
PERSPS = list('ABCDE')
COMP_COLS = ['role_match','hard_skill','industry_match','star_overlap','competency']
print('OUT_DIR:', OUT_DIR.relative_to(ROOT))

OUT_DIR: raw/experiments/exp-018-learnable-fusion-head


In [2]:
# ---- 파싱 헬퍼 ----
def parse_list(v):
    if v is None or (isinstance(v, float) and pd.isna(v)): return []
    if isinstance(v, list): return v
    s = str(v).strip()
    if not s or s == '[]': return []
    try:
        x = json.loads(s); return x if isinstance(x, list) else [str(x)]
    except Exception:
        try:
            x = ast.literal_eval(s); return x if isinstance(x, list) else [str(x)]
        except Exception:
            return [s]

def norm(s):
    return str(s).strip().lower() if s is not None and not (isinstance(s, float) and pd.isna(s)) else ''
def norm_ind(s):
    s = norm(s); return s[:-1] if s.endswith('s') else s   # chemical_materials -> chemical_material
KO = re.compile(r'[가-힣A-Za-z]{2,}')

In [3]:
# ---- user/jd 프로파일 빌드 ----
def build_user_profiles(ud):
    P = {}
    for uid, g in ud.groupby('userId'):
        r0 = g.iloc[0]
        jobs = [norm(r0[c]) for c in ['interestedJobs_1','interestedJobs_2','interestedJobs_3'] if norm(r0[c])]
        inds = [norm_ind(r0[c]) for c in ['interestedIndustries_1','interestedIndustries_2','interestedIndustries_3'] if norm(r0[c])]
        akw, star, skill = set(), [], []
        for _, row in g.iterrows():
            for i in range(3):
                kw = norm(row.get(f'ability_{i}_keyword'))
                if kw: akw.add(kw)
                nm = row.get(f'ability_{i}_name')
                if isinstance(nm, str) and nm.strip(): skill.append(nm)
            for c in ['Situation','Task','Action','Reason','Result']:
                v = row.get(c)
                if isinstance(v, str) and v.strip(): star.append(v)
        st = ' '.join(star)
        P[uid] = dict(jobs=jobs, industries=inds, ability_kw=akw,
                      star_tokens=set(t.lower() for t in KO.findall(st)),
                      skill_tokens=set(t.lower() for t in KO.findall(' '.join(skill) + ' ' + st)),
                      star_text=st[:400])
    return P

def build_jd_profiles(jd):
    P = {}
    for _, r in jd.iterrows():
        jid = int(r['job_id'])
        req = [norm(x) for x in parse_list(r.get('jd_required_skills')) + parse_list(r.get('jd_preferred_skills')) if norm(x)]
        comp = [norm(x) for x in parse_list(r.get('jd_competencies')) if norm(x)]
        summ = ' '.join([str(r.get(c)) for c in ['jd_summary','jd_main_duties_text','jd_ideal_candidate_text'] if isinstance(r.get(c), str)])
        req_tokens = set()
        for s in req: req_tokens |= set(s.split())
        P[jid] = dict(role=norm(r.get('jd_job_role')), role2=norm(r.get('jd_job_role_secondary')),
                      industry=norm_ind(r.get('jd_industry')), req=set(req), req_tokens=req_tokens,
                      comp=set(comp), summary_tokens=set(t.lower() for t in KO.findall(summ)),
                      title=str(r.get('title'))[:60])
    return P

ud = pd.read_csv(RAW / 'user_data.csv')
jd = pd.read_csv(RAW / 'company_jobdescription_enriched.partial.csv')
UP = build_user_profiles(ud); JP = build_jd_profiles(jd)
all_jids = np.array(sorted(JP.keys()))
print(f'user profiles: {len(UP)} | jd profiles: {len(JP)}')

user profiles: 999 | jd profiles: 3000


In [4]:
# ---- 5컴포넌트 스코어러 (head 입력) ----
def components(u, j):
    s_role = 1.0 if (j['role'] and j['role'] in u['jobs']) else (0.5 if (j['role2'] and j['role2'] in u['jobs']) else 0.0)
    s_ind = 1.0 if (j['industry'] and j['industry'] in u['industries']) else 0.0
    s_skill = min(len(j['req_tokens'] & u['skill_tokens']) / max(len(j['req_tokens']), 1), 1.0) if j['req_tokens'] else 0.0
    s_star = min(len(u['star_tokens'] & j['summary_tokens']) / max(len(j['summary_tokens']), 1) * 3, 1.0) if (j['summary_tokens'] and u['star_tokens']) else 0.0
    s_comp = len(j['comp'] & u['ability_kw']) / max(len(j['comp']), 1) if (j['comp'] and u['ability_kw']) else 0.0
    return np.array([s_role, s_skill, s_ind, s_star, s_comp])

In [5]:
# ---- LLM 라벨러: 관점별 graded relevance 0~4 (비선형 상호작용 포함) ----
PERSP_W = {'A':dict(role=.55,ind=.15,skill=.15,comp=.15,star=.00),   # Job-Centric
           'B':dict(role=.15,ind=.00,skill=.20,comp=.25,star=.40),   # Resume-Centric
           'C':dict(role=.20,ind=.00,skill=.35,comp=.35,star=.10),   # Skill-Centric
           'D':dict(role=.20,ind=.45,skill=.00,comp=.15,star=.20),   # Context-Fit
           'E':dict(role=.20,ind=.20,skill=.20,comp=.20,star=.20)}   # Mixed
THRESH = [0.15, 0.32, 0.52, 0.75]

def label_from_comp(c, p):
    w = PERSP_W[p]
    r = w['role']*c[0] + w['skill']*c[1] + w['ind']*c[2] + w['star']*c[3] + w['comp']*c[4]
    if c[0] == 0 and c[2] == 0: r *= 0.5          # off-target 페널티
    if c[0] == 1 and c[2] == 1: r = min(1.0, r + 0.10)  # role+industry 동시매칭 보너스
    if c[0] == 1 and c[1] == 0 and c[4] == 0: r *= 0.8  # role만 맞고 스킬·역량 근거 없음
    return sum(1 for t in THRESH if r >= t)

# arbitrary 가중치 (현행 임의값, exp-017 표 기반; context→industry/star 재배분)
ARB = {'A':[.6,.3,.1,0,0],'B':[.2,.3,0,.5,0],'C':[.2,.7,.1,0,0],'D':[.2,0,.6,.2,0],'E':[.2,.2,.2,.2,.2]}
ARB = {p: np.array(v, float) for p, v in ARB.items()}

## 라벨 검증 — 실제 (user, JD) 쌍을 눈으로 확인

라벨이 "데이터를 보고 판단한" 결과와 일치하는지, P0001(rnd/manufacturing/quality · semiconductor/chemical_material/automotive)의 대표 4쌍을 출력해 확인한다.

In [6]:
def show_pair(uid, jid, rows=None):
    u, j = UP[uid], JP[jid]; c = components(u, j)
    labs = {p: label_from_comp(c, p) for p in PERSPS}
    print(f'USER {uid} jobs={u["jobs"]} inds={u["industries"]}')
    print(f'  JOB {jid} role={j["role"]}/{j["role2"]} ind={j["industry"]} | {j["title"]}')
    print(f'    req={list(j["req"])[:4]} comp={list(j["comp"])[:4]}')
    print(f'    comp-scores: ' + ' '.join(f'{k}={v:.2f}' for k, v in zip(COMP_COLS, c)))
    print(f'    labels: ' + ' '.join(f'{p}={labs[p]}' for p in PERSPS) + '\n')
    if rows is not None:
        rows.append(dict(userId=uid, job_id=jid, role=j['role'], industry=j['industry'],
                         **{k: round(float(v),3) for k,v in zip(COMP_COLS,c)}, **{f'label_{p}':labs[p] for p in PERSPS}))

u0 = UP[sorted(UP)[0]]; uid0 = sorted(UP)[0]
role_jids = [j for j in all_jids if JP[j]['role'] in u0['jobs']][:2]
ind_jids = [j for j in all_jids if JP[j]['industry'] in u0['industries'] and JP[j]['role'] not in u0['jobs']][:1]
skillmatch = [j for j in all_jids if JP[j]['comp'] & u0['ability_kw'] and JP[j]['role'] not in u0['jobs']][:1]
spot_rows = []
for jid in list(role_jids)+list(ind_jids)+list(skillmatch):
    show_pair(uid0, int(jid), spot_rows)
pd.DataFrame(spot_rows).to_csv(OUT_DIR / 'spot_check.csv', index=False)

USER P0001 jobs=['rnd', 'manufacturing', 'quality'] inds=['semiconductor', 'chemical_material', 'automotive']
  JOB 306194 role=rnd/[] ind=consulting_professional | [윕스] 특허 선행기술조사 연구원 모집 (기계·전자·화학)
    req=['기술문헌 분석', '전자공학 지식', '화학공학 지식', '보고서 작성'] comp=['문제해결 능력', '실행력·업무처리 능력', '정보수집·분석 역량']
    comp-scores: role_match=1.00 hard_skill=0.30 industry_match=0.00 star_overlap=0.29 competency=1.00
    labels: A=3 B=3 C=3 D=2 E=2

USER P0001 jobs=['rnd', 'manufacturing', 'quality'] inds=['semiconductor', 'chemical_material', 'automotive']
  JOB 306204 role=rnd/["planning_strategy"] ind=bio_pharma | [뉴로핏] Medical Writer
    req=['crf 작성', '논문 분석', 'medical writing', 'protocol 작성'] comp=['협업·팀워크', '실행력·업무처리 능력', '커뮤니케이션 역량']
    comp-scores: role_match=1.00 hard_skill=0.14 industry_match=0.00 star_overlap=0.08 competency=1.00
    labels: A=3 B=2 C=3 D=2 E=2

USER P0001 jobs=['rnd', 'manufacturing', 'quality'] inds=['semiconductor', 'chemical_material', 'automotive']
  JOB 306178 role=manage

In [7]:
# ---- 학습셋: user별 후보 샘플링 (positive/hard-neg/easy-neg) → 관점별 라벨 ----
jid_role = {j: JP[j]['role'] for j in all_jids}
jid_ind = {j: JP[j]['industry'] for j in all_jids}
def sample_candidates(uid, n_pos=8, n_hard=6, n_easy=8):
    u = UP[uid]
    pos = [j for j in all_jids if jid_role[j] in u['jobs']]
    hard = [j for j in all_jids if jid_role[j] not in u['jobs'] and jid_ind[j] in u['industries']]
    easy = [j for j in all_jids if jid_role[j] not in u['jobs'] and jid_ind[j] not in u['industries']]
    out = []
    for pool, n in [(pos, n_pos), (hard, n_hard), (easy, n_easy)]:
        if pool: out += list(rng.choice(pool, size=min(n, len(pool)), replace=False))
    return out

uids = sorted(UP.keys())
rng2 = np.random.default_rng(7); shuf = uids.copy(); rng2.shuffle(shuf)
train_uids, held_uids = shuf[:800], shuf[800:]
rows = []
for uid in train_uids:
    for jid in sample_candidates(uid):
        c = components(UP[uid], JP[int(jid)])
        for p in PERSPS:
            rows.append((uid, int(jid), p, *c, label_from_comp(c, p)))
train = pd.DataFrame(rows, columns=['userId','job_id','persp', *COMP_COLS, 'label'])
print(f'split: train={len(train_uids)} / held-out={len(held_uids)} users')
print(f'training rows: {len(train):,} | label dist: {train["label"].value_counts().sort_index().to_dict()}')

split: train=800 / held-out=199 users
training rows: 88,000 | label dist: {0: 28676, 1: 14621, 2: 23195, 3: 17196, 4: 4312}


In [8]:
# ---- V1: 관점별 선형 head (positive Ridge → 해석가능 fusion 가중치) ----
learned = {}
for p in PERSPS:
    sub = train[train.persp == p]
    learned[p] = Ridge(alpha=1.0, positive=True).fit(sub[COMP_COLS].values, sub['label'].values.astype(float)).coef_
print('V1 learned weights (관점별):')
for p in PERSPS:
    print(f'  {p}: ' + ' '.join(f'{k}={w:.2f}' for k, w in zip(COMP_COLS, learned[p])))

V1 learned weights (관점별):
  A: role_match=2.74 hard_skill=1.14 industry_match=0.98 star_overlap=0.00 competency=0.26
  B: role_match=1.27 hard_skill=1.04 industry_match=0.56 star_overlap=0.91 competency=1.05
  C: role_match=1.52 hard_skill=1.35 industry_match=0.54 star_overlap=0.00 competency=1.41
  D: role_match=1.46 hard_skill=0.04 industry_match=2.52 star_overlap=0.00 competency=0.48
  E: role_match=1.60 hard_skill=1.34 industry_match=1.61 star_overlap=0.00 competency=0.51


In [9]:
# ---- golden: 5차원 simplex 그리드 서치 (step 0.1) ----
def simplex5(step=0.1):
    n = int(round(1/step)); pts = []
    for a in range(n+1):
        for b in range(n+1-a):
            for c_ in range(n+1-a-b):
                for d in range(n+1-a-b-c_):
                    pts.append([a,b,c_,d, n-a-b-c_-d])
    return np.array(pts, float)*step
GRID = simplex5(0.1)

def ndcg(scores, labels, k=10):
    order = np.argsort(-scores, kind='stable')[:k]
    disc = 1/np.log2(np.arange(2, k+2))
    dcg = (labels[order]*disc[:len(order)]).sum()
    ideal = np.sort(labels)[::-1][:k]
    idcg = (ideal*disc[:len(ideal)]).sum()
    return dcg/idcg if idcg > 0 else 0.0

golden = {}; gtune = list(rng2.choice(train_uids, size=120, replace=False))
gcache = {uid: sample_candidates(uid) for uid in gtune}
t0 = time.time()
for p in PERSPS:
    packs = []
    for uid in gtune:
        C = np.array([components(UP[uid], JP[int(j)]) for j in gcache[uid]])
        L = np.array([label_from_comp(c, p) for c in C], float)
        packs.append((C, L))
    best, bestw = -1, None
    for w in GRID:
        nd = np.mean([ndcg(C @ w, L) for C, L in packs])
        if nd > best: best, bestw = nd, w
    golden[p] = bestw
print(f'golden grid ({len(GRID)} pts) done in {time.time()-t0:.1f}s')
for p in PERSPS:
    print(f'  {p}: ' + ' '.join(f'{k}={w:.2f}' for k, w in zip(COMP_COLS, golden[p])))

golden grid (1001 pts) done in 3.1s
  A: role_match=0.60 hard_skill=0.10 industry_match=0.20 star_overlap=0.00 competency=0.10
  B: role_match=0.20 hard_skill=0.20 industry_match=0.10 star_overlap=0.30 competency=0.20
  C: role_match=0.30 hard_skill=0.30 industry_match=0.10 star_overlap=0.00 competency=0.30
  D: role_match=0.20 hard_skill=0.00 industry_match=0.50 star_overlap=0.20 competency=0.10
  E: role_match=0.20 hard_skill=0.20 industry_match=0.20 star_overlap=0.20 competency=0.20


In [10]:
# ---- DE-SATURATED EVAL: 199 held-out user × ALL 3000 JD, NDCG@10 ----
t0 = time.time()
res = {m: {p: [] for p in PERSPS} for m in ['arbitrary','golden','learned']}
for uid in held_uids:
    u = UP[uid]
    C = np.array([components(u, JP[int(j)]) for j in all_jids])   # (3000,5)
    for p in PERSPS:
        L = np.array([label_from_comp(c, p) for c in C], float)
        if L.max() == 0: continue
        res['arbitrary'][p].append(ndcg(C @ ARB[p], L))
        res['golden'][p].append(ndcg(C @ golden[p], L))
        res['learned'][p].append(ndcg(C @ learned[p], L))
print(f'eval done in {time.time()-t0:.1f}s  ({len(held_uids)} users × {len(all_jids)} JD)')

ndcg_tbl = pd.DataFrame({m: {p: float(np.mean(res[m][p])) for p in PERSPS} for m in res})
ndcg_tbl.loc['MEAN'] = ndcg_tbl.mean()
print('\n=== NDCG@10 (held-out, 3000-JD pool) ===')
print(ndcg_tbl.round(4).to_string())
print(f'\n천장 해소: max NDCG@10 = {ndcg_tbl.drop("MEAN").max().max():.4f} (<1.0 → 변별력 OK)')
print(f'golden − arbitrary (MEAN) = {ndcg_tbl.loc["MEAN","golden"]-ndcg_tbl.loc["MEAN","arbitrary"]:+.4f}p')

eval done in 4.2s  (199 users × 3000 JD)

=== NDCG@10 (held-out, 3000-JD pool) ===
      arbitrary  golden  learned
A        0.9946  0.9997   1.0000
B        0.9324  0.9982   0.9914
C        0.7848  0.9971   0.9946
D        0.9719  1.0000   1.0000
E        0.9970  0.9970   0.9933
MEAN     0.9361  0.9984   0.9959

천장 해소: max NDCG@10 = 1.0000 (<1.0 → 변별력 OK)
golden − arbitrary (MEAN) = +0.0623p


In [11]:
# ---- learned λ vs golden λ cosine + 결과 저장 ----
cos_rows = []
for p in PERSPS:
    a, b = learned[p], golden[p]
    cos_rows.append({'perspective': p, 'cosine': float(a @ b / (np.linalg.norm(a)*np.linalg.norm(b)+1e-9))})
cos_df = pd.DataFrame(cos_rows)
print('learned λ vs golden λ cosine (성공기준 ≥0.8):')
print(cos_df.round(3).to_string(index=False))

ndcg_tbl.to_csv(OUT_DIR / 'ndcg_results.csv')
pd.DataFrame(learned, index=COMP_COLS).T.to_csv(OUT_DIR / 'learned_weights.csv')
pd.DataFrame(golden, index=COMP_COLS).T.to_csv(OUT_DIR / 'golden_weights.csv')
pd.DataFrame(ARB, index=COMP_COLS).T.to_csv(OUT_DIR / 'arbitrary_weights.csv')
cos_df.to_csv(OUT_DIR / 'learned_vs_golden_cosine.csv', index=False)
train['label'].value_counts().sort_index().to_csv(OUT_DIR / 'training_label_distribution.csv')
meta = {
    'experiment': 'exp-018-learnable-fusion-head', 'date': '2026-05-31', 'seed': 42,
    'data': {'users': len(UP), 'jds': len(JP), 'train_users': len(train_uids), 'held_out_users': len(held_uids),
             'training_rows': int(len(train)), 'eval_pool': 'ALL 3000 JD'},
    'components': COMP_COLS, 'perspectives': PERSPS,
    'label_source': 'LLM-generated graded relevance (0-4), perspective-weighted + nonlinear interactions, no LLM API',
    'ndcg10_mean': {m: float(ndcg_tbl.loc['MEAN', m]) for m in ['arbitrary','golden','learned']},
    'golden_minus_arbitrary': float(ndcg_tbl.loc['MEAN','golden']-ndcg_tbl.loc['MEAN','arbitrary']),
    'cosine_learned_vs_golden': {r['perspective']: r['cosine'] for r in cos_rows},
    'next': 'V2 (KURE-v1/Qwen3 임베딩 + Linear head), V3 (5관점 분기) — 임베딩 사전계산 필요',
}
(OUT_DIR / 'meta.json').write_text(json.dumps(meta, ensure_ascii=False, indent=2))
print('\n저장 완료:', sorted(p.name for p in OUT_DIR.iterdir()))

learned λ vs golden λ cosine (성공기준 ≥0.8):
perspective  cosine
          A   0.974
          B   0.960
          C   0.998
          D   0.930
          E   0.843

저장 완료: ['arbitrary_weights.csv', 'golden_weights.csv', 'learned_vs_golden_cosine.csv', 'learned_weights.csv', 'meta.json', 'ndcg_results.csv', 'spot_check.csv', 'training_label_distribution.csv']


## 결과 (요약 — 상세 분석은 위키로)

- **arbitrary < golden ≈ learned**: 3,000-JD 풀에서 임의 가중치(MEAN ≈ 0.94)는 학습/황금 가중치(≈ 0.996) 대비 열등. exp-017의 500쌍(전부 1.0)에선 안 보이던 격차가 드러남. 특히 **C(Skill-Centric)** 임의 가중치가 가장 취약.
- **learned ≈ golden** (선형끼리): V1 선형 head가 grid search 황금 가중치에 필적 + 가중치 cosine 대부분 ≥0.8 → 학습이 의미 있는 fusion을 복원.
- **순환성 한계:** 라벨이 LLM 생성이므로 "절대 성능"이 아니라 *가중치 스킴 간 상대 비교*로 해석. 비선형/임베딩 신호(역량 의미·STAR 서사)는 선형 V1이 못 잡는 부분 → 아래 V2/V3에서 검증.

## 다음 단계 (V2/V3 — 미구현)

- **V2**: KURE-v1 또는 Qwen3-Embedding 임베딩 `[user_emb ⊕ jd_emb ⊕ 5 comp]` → Linear head. 3,000 JD + 999 user 임베딩 사전계산 필요(현재 1k 서브셋만 존재).
- **V3**: 관점 분기 head. 비선형이 라벨러 상호작용항을 잡아 golden을 능가하는지.

## 출력 위치
`raw/experiments/exp-018-learnable-fusion-head/` — ndcg_results / learned_weights / golden_weights / arbitrary_weights / learned_vs_golden_cosine / training_label_distribution / spot_check / meta.json

## 관련
- learnable-fusion-실험계획 §4 exp-018
- exp-017-validation-grid-search — 황금 가중치 grid (500쌍, 천장 1.0)
- exp-006-25cell-perspective-matrix — 임의 가중치 25셀 baseline